## Tools in LLMs

Tools allow an AI model to interact with external systems and perform tasks beyond generating text, such as searching the web, fetching data from a database, calling APIs, or executing code.

Tools are especially important in AI agents, where the model decides when and how to use a tool to accomplish a task.

### Components of a Tool

Tools consist of two main parts:

1. Tool Schema: Defines the tool's name, description, and input arguments, often using a JSON schema. It tells the model what the tool does and what inputs it requires.
2. Tool Implementation: The actual function or coroutine that executes the task when called.

### How Tool Calling Works

1. The developer provides the model with available tool definitions.
2. The model decides whether a tool is needed for the user's request.
3. The model generates a tool call with the required arguments.
4. The application executes the corresponding function.
5. The tool returns its result to the model.
6. The model uses the result to generate a response for the user.

### Example

Suppose an AI assistant has access to a weather tool.

- Tool name: `get_weather`
- Description: Fetches the current weather for a specified city.
- Input: `city = "Gorakhpur"`
- Implementation: Calls a weather API and retrieves the current conditions.
- Output: Weather information that the model can use to answer the user.

Important points:

- The model typically requests a tool call; the application or agent framework executes it.
- A tool call is different from an ordinary text response.
- Tools can connect LLMs to APIs, databases, search engines, and other software.
- Tool calling is a key capability used in agentic AI, but using tools alone does not automatically make an application a fully autonomous agent.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("Why do parrots talk?")
response.content

'Parrots aren’t really “talking” the way humans do, but they are incredibly good at **mimicking** sounds they hear—including human speech. Their ability to do this comes from a combination of anatomy, brain wiring, and social behavior.\n\n### 1. Anatomy that makes mimicry possible  \n\n| Feature | What it does | Why it matters for speech |\n|---------|--------------|---------------------------|\n| **Syrinx** (bird’s voice box) | Located at the base of the trachea, it can produce a huge variety of tones by vibrating membranes. | Allows very fine control of pitch, volume, and timbre—much more flexible than a human larynx. |\n| **Tongue & beak** | Highly mobile tongue and a beak that can shape sounds. | Helps shape the exact “words” they imitate. |\n| **Auditory memory** | Parrots have excellent hearing and can store complex sound patterns. | They can recall and reproduce phrases they’ve heard repeatedly. |\n\n### 2. Brain wiring and learning style  \n\n- **Songbird‑like brain regions** –

In [44]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the live weather of the given location."""
    return f"The current weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])



In [45]:
response = model_with_tools.invoke(["What's the weather like in London?"])
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for the weather in London. We have a function get_weather. Use it.', 'tool_calls': [{'id': 'fc_60a7e841-92d2-464d-85ca-6e5e8634ef1f', 'function': {'arguments': '{"location":"London"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 129, 'total_tokens': 176, 'completion_time': 0.098469938, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005084334, 'prompt_tokens_details': None, 'queue_time': 0.212898026, 'total_time': 0.103554272}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18309-2a98-7103-9b76-afd94895670a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'London'}, 'id': 'fc_60a7e841-92d2-464d-85ca-6e5e8634ef1f', 'type': 'tool_call'}], invalid_tool_

In [46]:
for tool_call in response.tool_calls:
    # View the tool calls made by the modell
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'London'}


### Tool Execution Loops

In [47]:
# Step1 : Model generates tool calls
messages = [{"role":"user","content":"What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston right now is sunny.


In [48]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. We need to call get_weather function.', 'tool_calls': [{'id': 'fc_4911a8ae-1931-4d71-ba45-b9012ac8059f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 129, 'total_tokens': 172, 'completion_time': 0.094872643, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.00625157, 'prompt_tokens_details': None, 'queue_time': 0.212722549, 'total_time': 0.101124213}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18309-2c2e-77b1-b057-e608abfa0abb-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_4911a8ae-1931-4d71-ba